In [1]:
import pandas as pd
import os
import numpy as np
import polars as pl
import json
import pygwalker as pyg
from glob import glob
import gc

In [2]:
# --- 경로 설정 ---
root_path = r'C:\Users\fienn\Documents\project\da_team_project_4\data\github_bckup\0_data'
filtered_match_detail_path = os.path.join(root_path, '17_match_details', 'participants')
filtered_match_detail_files = glob(os.path.join(filtered_match_detail_path, 'details_*_participants.parquet'))

# 마스터 match_id CSV 파일 경로
master_id_path = os.path.join(root_path, '16_matchids', 'valid_matchids_all_platforms.csv')

df = pd.read_parquet(os.path.join(filtered_match_detail_path, 'global_match_details_participants.parquet'))

In [3]:
# ==========================================
# 1. 유효한 포지션만 필터링 및 기초 집계
# ==========================================
valid_positions = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
df_clean = df[df['team_position'].isin(valid_positions)].copy()

# puuid별 포지션 플레이 횟수 및 총 횟수 집계
pos_counts = df_clean.groupby(['puuid', 'team_position']).size().reset_index(name='play_count')
total_counts = df_clean.groupby('puuid').size().reset_index(name='total_games')

# 표본이 적은 유저 제외 (5판 이상)
valid_users = total_counts[total_counts['total_games'] >= 5]
pos_counts = pos_counts[pos_counts['puuid'].isin(valid_users['puuid'])]

# ==========================================
# 2. 수학적 지표 (집중도 & 다양성) 벡터 연산
# ==========================================
df_pos = pd.merge(pos_counts, valid_users, on='puuid')
df_pos['ratio'] = df_pos['play_count'] / df_pos['total_games']

df_pos['hhi_term'] = df_pos['ratio'] ** 2
df_pos['entropy_term'] = df_pos['ratio'] * np.log(df_pos['ratio'])

metrics_df = df_pos.groupby('puuid').agg(
    concentration_index=('hhi_term', 'sum'),
    diversity_index=('entropy_term', lambda x: -x.sum())
).reset_index()

# ==========================================
# 3. [개선됨] 랭킹 기반 주력(Main) 및 서브(Sub) 포지션 추출
# ==========================================
# 플레이 횟수를 기준으로 유저별 포지션 랭킹 부여 (동점일 경우 임의 순위 배정)
df_pos['pos_rank'] = df_pos.groupby('puuid')['play_count'].rank(method='first', ascending=False)

# 1순위 포지션을 Main, 2순위 포지션을 Sub로 추출
main_df = df_pos[df_pos['pos_rank'] == 1][['puuid', 'team_position']].rename(columns={'team_position': 'main_position'})
sub_df = df_pos[df_pos['pos_rank'] == 2][['puuid', 'team_position']].rename(columns={'team_position': 'sub_position'})

# ==========================================
# 4. 최종 데이터 마트 조립 (Merge)
# ==========================================
user_profile_df = valid_users.merge(metrics_df, on='puuid', how='left') \
                             .merge(main_df, on='puuid', how='left') \
                             .merge(sub_df, on='puuid', how='left')

# NaN 값을 'None'으로 깔끔하게 채우기 (한 포지션만 파는 원챔 유저는 Sub가 없을 수 있음)
user_profile_df['main_position'] = user_profile_df['main_position'].fillna('None')
user_profile_df['sub_position'] = user_profile_df['sub_position'].fillna('None')

user_profile_df['concentration_index'] = user_profile_df['concentration_index'].round(3)
user_profile_df['diversity_index'] = user_profile_df['diversity_index'].round(3)

print(f"✅ 유저 프로파일 생성 완료! (대상: {len(user_profile_df):,}명)")

# ==========================================
# 5. 원본 데이터(df)에 프로파일 결합 및 오토필 태깅
# ==========================================
# 원본 매치 데이터에 프로필의 Main/Sub 포지션 정보 결합
df = df.merge(user_profile_df[['puuid', 'main_position', 'sub_position']], on='puuid', how='left')

col_team = 'team_position'
col_ind = 'individual_position'

unwanted_status_list = []

for t_pos, i_pos, m_pos, s_pos in zip(df[col_team], df[col_ind], df['main_position'], df['sub_position']):
    t_pos_str = str(t_pos)
    i_pos_str = str(i_pos)
    m_pos_str = str(m_pos)
    s_pos_str = str(s_pos)
    
    # [추가됨] 1. 애초에 데이터가 부족해 프로필이 없는 유저 (통계에서 제외하기 위함)
    if m_pos_str == 'nan':
        unwanted_status_list.append('Unknown_Profile')
        
    # [목적 달성] 2. 1순위(Main) 또는 2순위(Sub) 포지션을 플레이했다면 정상
    # elif i_pos_str != 'nan' and i_pos_str != 'None' and (i_pos_str == m_pos_str or i_pos_str == s_pos_str):
    #     unwanted_status_list.append('Played_Desired')
    
    # 2-2. t_pos_str 기준으로 실험
    elif t_pos_str != 'nan' and t_pos_str != 'None' and (t_pos_str == m_pos_str or t_pos_str == s_pos_str):
        unwanted_status_list.append('Played_Desired')
        
    # [강제 오토필] 3. 주력/서브가 아닌데, 시스템이 꽂아넣은 자리를 그대로 플레이함
    elif i_pos_str == t_pos_str:
        unwanted_status_list.append('Forced_AutoFill')
        
    # [희생/양보] 4. 주력/서브가 아닌데, 원래 배정과도 다른 자리를 플레이함 (스왑)
    else:
        unwanted_status_list.append('Sacrificed_Swap')

df['unwanted_play_status'] = unwanted_status_list

# 임시 메모리 정리
del df_clean, total_counts, valid_users, df_pos, metrics_df, main_df, sub_df
gc.collect()

# 결과 재확인 ('Unknown_Profile'을 제외하고 순수하게 판별된 유저들 사이의 비율만 계산)
known_df = df[df['unwanted_play_status'] != 'Unknown_Profile']
status_counts_fixed = known_df['unwanted_play_status'].value_counts(normalize=True) * 100

print("📊 [오류 수정] 식별 가능한 유저 기준 포지션 만족도 비율:")
print(status_counts_fixed.round(2).astype(str) + '%')

✅ 유저 프로파일 생성 완료! (대상: 8,598명)
📊 [오류 수정] 식별 가능한 유저 기준 포지션 만족도 비율:
unwanted_play_status
Played_Desired     85.61%
Forced_AutoFill    14.23%
Sacrificed_Swap     0.16%
Name: proportion, dtype: object


In [4]:
# ==========================================
# 1. 분석 대상 그룹 매핑 (A/B 테스트 세팅)
# ==========================================
# 앞서 'Unknown_Profile'을 제외하고 필터링해 둔 known_df를 사용합니다.
target_groups = ['Played_Desired', 'Forced_AutoFill', 'Sacrificed_Swap']
analysis_df = known_df[known_df['unwanted_play_status'].isin(target_groups)].copy()

# 두 개의 큰 그룹으로 묶어줍니다.
analysis_df['group'] = np.where(
    analysis_df['unwanted_play_status'] == 'Played_Desired',
    'Desired (주력/서브)',
    'Auto-filled (오토필)'
)

# ==========================================
# 2. 파생 지표 생성 (KDA, 서렌)
# ==========================================
# 데스가 0일 경우 무한대(Inf) 에러가 발생하므로 1로 보정하여 계산합니다. (Perfect KDA 처리)
analysis_df['deaths_safe'] = analysis_df['deaths'].replace(0, 1)
analysis_df['kda'] = (analysis_df['kills'] + analysis_df['assists']) / analysis_df['deaths_safe']

# 승리 여부, 서렌 여부 정수형 변환 (이미 되어있다면 생략 가능)
analysis_df['win_int'] = analysis_df['win'].astype(int)
analysis_df['game_ended_in_surrender'] = analysis_df['game_ended_in_surrender'].astype(int)
analysis_df['game_ended_in_early_surrender'] = analysis_df['game_ended_in_early_surrender'].astype(int)
analysis_df['team_early_surrendered'] = analysis_df['team_early_surrendered'].astype(int)

# ==========================================
# 3. 그룹별 주요 성과 지표(KPI) 집계
# ==========================================
comparison_metrics = analysis_df.groupby('group').agg(
    total_games=('match_id', 'count'),
    win_rate=('win_int', lambda x: (x.mean() * 100)),
    avg_kda=('kda', 'mean'),
    avg_kills=('kills', 'mean'),
    avg_deaths=('deaths', 'mean'),
    avg_assists=('assists', 'mean'),
    avg_kp=('ch_kill_participation', 'mean'),
    avg_gold=('gold_earned', 'mean'),
    avg_damage=('total_damage_dealt_to_champions', 'mean'),
    avg_laning_phase_gold_exp_advantage=('ch_laning_phase_gold_exp_advantage', 'mean'),
    avg_cs=('cs', 'mean'),
    avg_game_duration=('game_duration', 'mean')
    
).reset_index()

# ==========================================
# 4. 결과 테이블 포맷팅 (가독성 향상)
# ==========================================
# 소수점 반올림 및 퍼센트 기호 추가
comparison_metrics['win_rate'] = comparison_metrics['win_rate'].round(1).astype(str) + '%'
for col in ['avg_kda', 'avg_kills', 'avg_deaths', 'avg_assists', 'avg_kp', 'avg_laning_phase_gold_exp_advantage', 'avg_cs', 'avg_game_duration']:
    comparison_metrics[col] = comparison_metrics[col].round(2)

for col in ['avg_gold', 'avg_damage']:
    comparison_metrics[col] = comparison_metrics[col].round(0).astype(int) # 골드와 딜량은 정수로

print("📊 [그룹별 성과 지표 비교]")
# print(comparison_metrics.to_string(index=False))
display(comparison_metrics)

📊 [그룹별 성과 지표 비교]


,group,total_games,win_rate,avg_kda,avg_kills,avg_deaths,avg_assists,avg_kp,avg_gold,avg_damage,avg_laning_phase_gold_exp_advantage,avg_cs,avg_game_duration
0,Auto-filled (오토필),11677,48.4%,2.92,5.95,6.68,8.39,0.45,11361,21775,0.10,145.60,1766.52
1,Desired (주력/서브),69495,50.7%,3.33,6.39,6.22,8.20,0.46,11710,22290,0.12,156.34,1769.01


In [5]:
from scipy import stats

# ==========================================
# 1. 승률 차이 검정 (Chi-Square Test)
# ==========================================
# 귀무가설: 포지션 만족도(Desired vs Auto-filled)와 승패는 연관이 없다.
contingency_table = pd.crosstab(analysis_df['group'], analysis_df['win_int'])

chi2, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)

print("🎯 [승률 차이 가설 검정 (Chi-Square)]")
print(f"P-value: {p_val_chi2:.5f}")
if p_val_chi2 < 0.05:
    print("👉 결론: 두 그룹 간의 승률 차이는 통계적으로 100% 유의미합니다. (오토필이 승패에 직접적인 영향을 미침)\n")
else:
    print("👉 결론: 승률 차이는 우연일 수 있으며 통계적으로 유의미하지 않습니다.\n")

# ==========================================
# 2. 연속형 KPI 차이 검정 (Welch's T-Test)
# ==========================================
# 귀무가설: 두 그룹 간의 KDA, Kills, Deaths 등의 평균 차이는 없다.
desired_group = analysis_df[analysis_df['group'] == 'Desired (주력/서브)']
autofill_group = analysis_df[analysis_df['group'] == 'Auto-filled (오토필)']

# 검정할 KPI 목록 (데이터의 실제 컬럼명과 맞는지 확인해주세요)
kpi_list = ['kda', 'kills', 'deaths', 'assists','ch_kill_participation', # KDA
            'total_damage_dealt_to_champions',  # 딜량
            'gold_earned', 'ch_laning_phase_gold_exp_advantage', 'cs',  # 성장
            'game_duration', 'team_early_surrendered'
]

test_results = []

for kpi in kpi_list:
    # 결측치(NaN)가 있으면 에러가 나므로 안전하게 제거 후 검정
    data_a = desired_group[kpi].dropna()
    data_b = autofill_group[kpi].dropna()
    
    # Welch's T-Test (equal_var=False 설정이 핵심 Quick Win)
    t_stat, p_val = stats.ttest_ind(data_a, data_b, equal_var=False)
    
    # 평균 계산 (직관적 비교를 위해)
    mean_a = data_a.mean()
    mean_b = data_b.mean()
    
    # 유의성 판단 (신뢰수준 95%)
    significance = "유의미함 (O)" if p_val < 0.05 else "무의미함 (X)"
    
    test_results.append({
        '지표 (KPI)': kpi,
        '원하는 포지션 평균': round(mean_a, 2),
        '오토필 평균': round(mean_b, 2),
        'T-Statistic': round(t_stat, 2),
        'P-value': p_val, # 과학적 표기법으로 출력될 수 있음
        '통계적 유의성': significance
    })

results_df = pd.DataFrame(test_results)

print("⚔️ [핵심 지표(KPI) 가설 검정 (Welch's T-Test)]")
display(results_df) # 주피터 환경
# print(results_df.to_string(index=False))

🎯 [승률 차이 가설 검정 (Chi-Square)]
P-value: 0.00000
👉 결론: 두 그룹 간의 승률 차이는 통계적으로 100% 유의미합니다. (오토필이 승패에 직접적인 영향을 미침)

⚔️ [핵심 지표(KPI) 가설 검정 (Welch's T-Test)]


,지표 (KPI),원하는 포지션 평균,오토필 평균,T-Statistic,P-value,통계적 유의성
0,kda,3.33,2.92,12.49,1.205686e-35,유의미함 (O)
1,kills,6.39,5.95,9.03,1.852399e-19,유의미함 (O)
2,deaths,6.22,6.68,-12.74,5.326213e-37,유의미함 (O)
3,assists,8.20,8.39,-2.91,3.569131e-03,유의미함 (O)
4,ch_kill_participation,0.46,0.45,7.53,5.525058e-14,유의미함 (O)
5,total_damage_dealt_to_champions,22290.40,21774.92,3.76,1.713206e-04,유의미함 (O)
6,gold_earned,11709.50,11361.05,8.01,1.244086e-15,유의미함 (O)
7,ch_laning_phase_gold_exp_advantage,0.12,0.10,4.16,3.146304e-05,유의미함 (O)
8,cs,156.34,145.60,13.00,2.041006e-38,유의미함 (O)
9,game_duration,1769.01,1766.52,0.47,6.356358e-01,무의미함 (X)


In [6]:
desired_group['main_position'].value_counts(normalize=True)*100

main_position
BOTTOM     24.076552
JUNGLE     20.214404
MIDDLE     19.162530
UTILITY    18.509245
TOP        18.037269
Name: proportion, dtype: float64

In [7]:
analysis_df[(analysis_df['group']=='Auto-filled (오토필)') & (analysis_df['team_position'].isin(valid_positions)) & (analysis_df['individual_position'].isin(valid_positions))][['individual_position', 'team_position']].value_counts(normalize=True)*100

individual_position  team_position
UTILITY              UTILITY          24.424519
TOP                  TOP              23.407190
MIDDLE               MIDDLE           20.889732
JUNGLE               JUNGLE           17.535994
BOTTOM               BOTTOM           13.302871
JUNGLE               UTILITY           0.172429
BOTTOM               UTILITY           0.094836
JUNGLE               TOP               0.077593
UTILITY              BOTTOM            0.043107
JUNGLE               MIDDLE            0.034486
MIDDLE               BOTTOM            0.008621
TOP                  UTILITY           0.008621
Name: proportion, dtype: float64

In [8]:
# 1. 오토필 그룹만 필터링
autofill_df = analysis_df[analysis_df['group'] == 'Auto-filled (오토필)']

# 2. 포지션 및 챔피언별 플레이 횟수 계산
champ_counts = autofill_df.groupby(['team_position', 'champion_name']).size().reset_index(name='play_count')

# 3. 포지션별 총 플레이 횟수 계산 (비율을 구하기 위해)
pos_total = autofill_df.groupby('team_position').size().reset_index(name='pos_total')

# 4. 데이터 병합 후 비율(%) 계산
champ_ratio = pd.merge(champ_counts, pos_total, on='team_position')
champ_ratio['pick_rate(%)'] = (champ_ratio['play_count'] / champ_ratio['pos_total'] * 100).round(2)

# 5. 포지션별로 가장 많이 픽한 챔피언 Top 5 추출
top5_autofill_champs = (
    champ_ratio.sort_values(by=['team_position', 'pick_rate(%)'], ascending=[True, False])
    .groupby('team_position')
    .head(5)
)
display(top5_autofill_champs)

,team_position,champion_name,play_count,pos_total,pick_rate(%)
19,,Lux,7,77,9.09
9,,DrMundo,3,77,3.90
0,,Aatrox,2,77,2.60
7,,Camille,2,77,2.60
8,,Darius,2,77,2.60
79,BOTTOM,Jhin,142,1549,9.17
93,BOTTOM,MissFortune,134,1549,8.65
67,BOTTOM,Caitlyn,132,1549,8.52
71,BOTTOM,Ezreal,105,1549,6.78
80,BOTTOM,Jinx,104,1549,6.71
